First Tool/Data Source

In [64]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [65]:
# Wikipedia blocks requests without a user agent (403), so set one
import wikipedia
wikipedia.wikipedia.USER_AGENT = "LangChainPractice/1.0 (learning project)"

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
tool1= [WikipediaQueryRun(api_wrapper=api_wrapper)]
tool1[0].name

'wikipedia'

Second Tool/ Data source

In [66]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [67]:
#load data from website 
loader= WebBaseLoader("https://docs.langchain.com/oss/python/langchain/tools")  
docs= loader.load()
#chuiing 
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
document= text_splitter.split_documents(docs)
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/tools', 'title': 'Tools - Docs by LangChain', 'description': 'Define tools that let LangChain agents fetch real-time data, execute code, query external databases, and take actions.', 'language': 'en'}, page_content='\n\n\n\n\nTools - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCore componentsToolsOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuard

In [69]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = Chroma.from_documents(document, embeddings)
retriever = db.as_retriever()
retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1588.76it/s]


VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001DBAB748B00>, search_kwargs={})

In [70]:
#creating tool of that retriver
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(retriever=retriever, name="LangchainTools", description="useful for when you need to answer questions about Langchain tools") 
retriever_tool.name

'LangchainTools'

In [71]:
## Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [72]:
#Combine all tools
tools= [tool1[0],retriever_tool,arxiv]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Tech Mehal\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=500)),
 StructuredTool(name='LangchainTools', description='useful for when you need to answer questions about Langchain tools', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001DBAE3D16F0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001DBAE3D1010>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200))]

In [73]:

# Free local LLM (Ollama) instead of paid OpenAI
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen2.5:3b", temperature=0, num_gpu=0)
llm.invoke("Say hello in one line").content

'Hello!'

In [75]:
#Prompt templete from Langchain hub
from langsmith import Client

prompt = Client().pull_prompt(
    "hwchase17/openai-functions-agent",
    dangerously_pull_public_prompt=True,
)
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')] | typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='Syst

In [76]:
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor

#creating agent & executor
agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='

In [ ]:
#Execute the tool with a query (Test1)
agent_executor.invoke({"input": "What is Langchain?"})



> Entering new AgentExecutor chain...

Invoking: `LangchainTools` with `{'query': 'Langchain'}`


Tools - Docs by LangChain

Tools - Docs by LangChain

Tools - Docs by LangChain

Tools - Docs by LangChainLangChain is a suite of tools designed for building and deploying machine learning models. It includes a variety of tools for text processing, data manipulation, model training, and more. The documentation for LangChain can be found [here](https://docs.langchain.com/).

If you need more specific information about a particular tool or feature, please provide me with more details.

> Finished chain.


{'input': 'What is Langchain?',
 'output': 'LangChain is a suite of tools designed for building and deploying machine learning models. It includes a variety of tools for text processing, data manipulation, model training, and more. The documentation for LangChain can be found [here](https://docs.langchain.com/).\n\nIf you need more specific information about a particular tool or feature, please provide me with more details.'}

In [80]:
#(Test2)
agent_executor.invoke({"input": "tell me about machine learning?"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed.
Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.
From Machine learning (ML) is a field of study in artificial intelligence that focuses on the development and study of statistical algorithms. These algorithms can learn from data and generalize to new, unseen data, thereby performing tasks without being explicitly programmed. The core of machine learning involves using statistical methods to analyze and interpret data, often from large datasets. It 

{'input': 'tell me about machine learning?',
 'output': 'Machine learning (ML) is a field of study in artificial intelligence that focuses on the development and study of statistical algorithms. These algorithms can learn from data and generalize to new, unseen data, thereby performing tasks without being explicitly programmed. The core of machine learning involves using statistical methods to analyze and interpret data, often from large datasets. It is closely related to data mining, which is more focused on the exploration of data for patterns and insights, often using unsupervised learning techniques. Machine learning is built on the foundations of statistics and mathematical optimization, and it plays a crucial role in various applications such as predictive analytics, pattern recognition, and decision-making processes.'}

In [81]:
#(Test3)
agent_executor.invoke({"input": "What is the paper 1605.08386 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1605.08386'}`


Published: 2016-05-26
Title: Heat-bath random walks with Markov bases
Authors: Caprice Stanley, Tobias Windisch
Summary: Graphs on lattice points are studied whose edges come from a finite set of alloThe paper with the identifier 1605.08386 is titled "Heat-bath random walks with Markov bases". It was published on 26th May 2016. The authors of this paper are Caprice Stanley and Tobias Windisch. The summary provided indicates that the paper discusses graphs on lattice points where edges are selected from a finite set of allowed edges.

> Finished chain.


{'input': 'What is the paper 1605.08386 about?',
 'output': 'The paper with the identifier 1605.08386 is titled "Heat-bath random walks with Markov bases". It was published on 26th May 2016. The authors of this paper are Caprice Stanley and Tobias Windisch. The summary provided indicates that the paper discusses graphs on lattice points where edges are selected from a finite set of allowed edges.'}